# Exploratory Data Analysis (EDA) - NuScenes & Surrogate Dataset

Ce notebook permet d'explorer les données brutes du dataset **NuScenes (v1.0-mini)** et les caractéristiques environnementales et comportementales extraites pour notre modèle de substitution (**Surrogate Model**).

## 1. Initialisation de NuScenes et Chargement des Librairies

In [ ]:
from nuscenes import NuScenes
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Détection du chemin vers les données
DATAROOT = '../data/sets' if os.path.exists('../data/sets') else 'data/sets'
nusc = NuScenes(version='v1.0-mini', dataroot=DATAROOT, verbose=True)

## 2. Exploration des Tables de Base (Scènes, Catégories, Annotations)

In [ ]:
# Liste des scènes du mini-kit
print(f"Nombre total de scènes dans le mini-kit : {len(nusc.scene)}\n")
for idx, scene in enumerate(nusc.scene):
    print(f"Scène {idx+1}: {scene['name']} - {scene['description']}")

In [ ]:
# Regarder les catégories d'objets annotées
categories = [ann['category_name'] for ann in nusc.sample_annotation]
cat_counts = pd.Series(categories).value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=cat_counts.values, y=cat_counts.index, hue=cat_counts.index, palette='viridis', legend=False)
plt.title("Distribution des catégories d'objets annotés dans NuScenes v1.0-mini")
plt.xlabel("Nombre d'annotations")
plt.ylabel("Catégorie")
plt.tight_layout()
plt.show()

## 3. Exploration du Dataset Extrait (data/extracted_dataset.csv)

Nous analysons ici le dataset extrait pour notre modèle de substitution. Ce fichier contient à la fois les variables d'environnement (vitesse, distance d'obstacle, courbure de voie) et les variables cibles (accélération et taux de changement de cap).

In [ ]:
csv_path = '../data/extracted_dataset.csv' if os.path.exists('../data/extracted_dataset.csv') else 'data/extracted_dataset.csv'
df = pd.read_csv(csv_path)
print(f"Nombre d'échantillons extraits : {df.shape[0]}")
print(f"Nombre de features : {df.shape[1]}")
df.head()

In [ ]:
df.describe()

## 4. Analyse des Distributions des Variables Cibles (Comportement du Véhicule)

In [ ]:
# Distribution des cibles : Braquage (Heading Rate) et Accélération
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['target_heading_rate'], kde=True, ax=axes[0], color='purple')
axes[0].set_title('Taux de Changement de Cap (target_heading_rate) [rad/s]')
axes[0].set_xlabel('Braquage (rad/s)')

sns.histplot(df['target_acceleration'], kde=True, ax=axes[1], color='teal')
axes[1].set_title("Accélération (target_acceleration) [m/s²]")
axes[1].set_xlabel('Accélération (m/s²)')

plt.tight_layout()
plt.show()

## 5. Analyse des Distributions des Variables Environnementales (Features)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

features = ['velocity', 'nearest_obstacle_distance', 'obstacles_within_10m', 'lane_curvature', 'distance_along_lane', 'distance_to_lane_end']
colors = ['royalblue', 'orange', 'forestgreen', 'crimson', 'mediumpurple', 'darkgoldenrod']

for idx, feat in enumerate(features):
    sns.histplot(df[feat], kde=True, ax=axes[idx], color=colors[idx])
    axes[idx].set_title(f'Distribution de {feat}')
    axes[idx].set_xlabel(feat)

plt.tight_layout()
plt.show()

## 6. Analyse des Corrélations

Observons comment les features géométriques et dynamiques corrèlent entre elles et avec les décisions du véhicule.

In [ ]:
plt.figure(figsize=(10, 8))
corr = df.corr()
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f", square=True, linewidths=.5)
plt.title("Matrice de Corrélation (Features & Cibles)")
plt.show()

## 7. Visualisations Jointes & Comportementales

In [ ]:
# Vitesse vs Accélération en fonction de la distance à l'obstacle
plt.figure(figsize=(12, 6))
scatter = plt.scatter(df['velocity'], df['target_acceleration'], 
                        c=df['nearest_obstacle_distance'], cmap='viridis_r', alpha=0.8)
cbar = plt.colorbar(scatter)
cbar.set_label("Distance à l'obstacle proche (m)")
plt.title("Relation Vitesse vs Accélération colorée par Distance d'Obstacle")
plt.xlabel("Vitesse (velocity) [m/s]")
plt.ylabel("Accélération (target_acceleration) [m/s²]")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

In [ ]:
# Courbure de la voie vs Taux de changement de cap (Braquage)
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='lane_curvature', y='target_heading_rate', hue='velocity', palette='magma')
plt.title("Courbure de la Voie vs Braquage (Heading Rate)")
plt.xlabel("Courbure locale de la Voie (lane_curvature)")
plt.ylabel("Braquage (target_heading_rate) [rad/s]")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()